# 05 Plan & Build

langgraph 로 agent 를 만들 때, 어떻게 생각하고 기획하는지. [참고 문서](https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph#draft-reply)

1. Node 로 단계 쪼개기 + Edge로 연결하기
2. 각 Node의 상세 작성
3. 공유하는 State 를 구성

## 고객 지원 이메일 담앙 agent: 기획 튜토리얼

Agent 요구사항:
1. 수신한 고객 이메일을 읽음
1. 급한 정도와 주제로 분류
1. 답변에 필요한 관련있는 문서를 검색
1. 적절한 대응 초안 작성
1. 복잡한 문제는 담당자에게 전달(Human In the Loop)
1. 필요시 후속 조치 예약

예시 사용자 시나리오:
- 단순 제품 질문: "전원 껐다키는 방법 뭐에요?"
- 버그 리포트: "pdf 추출했더니 꺼져요"
- 긴급 결제 이슈: "구독료 두번 나감!"
- 기능 추가 요청: "모바일에서 다크모드 만들어 주세요"
- 복잡한 기술적 문제: "API연동이 가끔 504 오류 뜨면서 실패함"

## 구현에 필요한 5 단계
1. 워크플로우를 개별 단계로 나눔
2. 각 단계에서 어떤 일을 처리할지 구체화
3. `State` 기획/디자인
4. 필요한 Node/Tool/Router 개발
5. 전체 조립

### 1. 워크플로우를 개별 단계로 나눔
- 그래프 디자인
```mermaid
flowchart TD

    START([START])
    READ[이메일 분석]
    CLASSIFY[의도 분류]

    DOC[문서 검색]
    BUG[버그 추적]
    HR[사람 리뷰]

    DRAFT[답장 초안]

    HR[사람 리뷰]
    SEND[답장 보내기]

    END([END])

    START --> READ
    READ --> CLASSIFY

    CLASSIFY -.-> |내부문서탐색| DOC
    CLASSIFY -.-> |버그리포트| BUG
    CLASSIFY -.-> |복잡한 문제| HR

    DOC --> DRAFT
    BUG --> DRAFT
    HR --> DRAFT

    DRAFT -.-> |긴급 높음| HR
    DRAFT -.-> |기타| SEND

    HR --> END
    SEND --> END
```

### 2. 각 단계(Node)에서 어떤 일을 처리할지 구체화
> 단일 노드(함수) | 단일 권한 | 단일 책임 

#### LLM 이 필요한 노드
- 의도 분류
    - Input (state): email 내용, 발신자 정보
    - Prompt: 카테고리 분류, 급한 단계, 응답 포맷
    - Output: 구조화된 분류 -> 다음 스텝 결정에 활용
- 초안 작성
    - Input: 분류 결과, 검색 결과, 고객 기록
    - Prompt: 어투 안내, 회사 정책, 답변 템플릿
    - Output: 전문적인 답변 메일 내용

#### Data (DB) 와 상호작용하는 노드
- 문서 검색
    - Parameter: 사용자 의도와 주제에 맞는 Query
    - Retry: 일시적 오류시, 0.5초-1초-2초-4초 후에 재시도 
    - Caching: 자주 사용하는 Query와 결과는 캐싱 가능

- 고객 기록 확인
    - Parameter: 사용자 email / ID
    - Retry: 시도, 만약 정보를 불러올 수 없으면 기본 정보로 대체
    - Caching: 정보의 실시간성을 위해 캐싱 유통기한 설정

#### API를 사용하는 노드
- 이메일 읽기
- 답장 보내기
    - When: 승인 이후 (사람/자동화)
    - Retry: 0.2, 0.4, 0.8, 1.6 초 마다 재시도
    - Return: Status Code(200, 400, 500)
- 버그 추적
    - When: 의도가 bug 로 파악된 경우 항상
    - Retry: 0.2, 0.4, 0.8, 1.6 초 마다 재시도
    - Return: 버그 이슈 티켓 ID

#### 사람 개입 노드 (HITL)
- 사람 리뷰
    - When: 매우 긴급, 복잡한 이슈, 답장 품질 우려
    - Context: 수신 이메일 원본, 답장 초안, 긴급 정도, 분류 카테고리
    - Human Output: O/X (보낸다 만다), 필요하다면 고친 내용

### 3. `State` 기획/디자인

> **어떤 정보를 `state` 에 저장할 것인가**
- 각 단계에 영구적으로 필요한 데이터인가? -> state 저장
- 다른 데이터에서 가져올 수 있는가? -> 그때 꺼내세요

**Email 에이전트에서 추적해야할 데이터** -> `state`
- `sender_email` : 발신자 이메일 주소
- `email_content` : 메일 내용

- `classification`: `intent`, `urgency`, `topic`, `summary`

- `search_results` : 문서 검색 결과
- `customer_history`: 사용자 상담 기록

- `draft_response`: 초안
- `messages` : agent가 step별로 생성한 모든 기록(메모리)

State 예시
```JSON
{
    "sender_email": "a@t.com",
    "email_content": "버그 발생",
    "classification": {
        "intent": "question" | "bug" | "billing" | "feature" | "complex",
        "urgency": "low" | "medium" | "high" | "critical",
        "topic": "???",
        "summary": "사용자가 결제에서 문제가 발생"
    },
    "search_results": ["결제오류는 ~~~처리한다", "결제가 이뤄지지 ~~~~~", "ㅇㄹㄹ이ㅏㅓ리ㅏㄴ"],
    "customer_history": {},
    "draft_response": "안녕하세요 고객님. 불편~~~",
    "messages": [HumanMessage, AIMessage, ToolMessage, ToolMessage, AIMessage, ...]
}
```


**중요한 원칙**
> `state` 는 날것의 데이터 (JSON, `dict`)를 그대로 사용. 자연어로 바꾸지 말것.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
# State
from typing import TypedDict, Literal
from langgraph.graph import MessagesState


# 1. 아래 State 형태를 결정할 때 사용
# 2. llm_with_output에서 스키마로 사용
class EmailClassification(TypedDict):
    intent: Literal['question', 'bug', 'billing', 'feature', 'complex']
    urgency: Literal['low', 'medium', 'high', 'critical']
    topic: str
    summary: str


class EmailAgentState(MessagesState):
    # messages 포함

    sender_email: str
    email_content: str

    classification: EmailClassification

    search_result = list[str] | None  # search_result 는 있으면 list[str], 없을 수도 있다.
    customer_history: dict | None

    draft_response: str | None

### 4. 필요한 Node/Tool/Router 개발

#### 이메일 읽기 & 분류 노드

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

llm = init_chat_model('openai:gpt-4.1-mini')


def classify_intent_node(state: EmailAgentState):
    '''LLM으로 이메일 분류: 의도와 긴급정도에 따라서 다음 노드가 결정되야함'''
    # 위에서 작성한 스키마 주입

    structured_llm = llm.with_structured_output(EmailClassification)
    prompt = f"""
    Analyze this customer email and classify it:

    Email: {state['email_content']}
    From: {state['sender_email']}

    Provide classification including intent, urgency, topic, and summary.
    """
    classification = structured_llm.invoke(prompt)

    # state의 classification을 갱신
    return {'classification': classification}


# 라우터
def intent_router(state: EmailAgentState):
    intent = state['classification']['intent']
    urgecy = state['classification']['urgency']
    # intent 가 billing. or 긴급도가 'critical' 이면, 'human_review' 리턴
    if intent == 'billing' or urgecy == 'critical':
        return 'human_review'
    # intent가 'question' 이나 'feature' 면, 'search_document' 리턴
    elif intent in ('question', 'feature'):
        return 'search_document'
    # intent 가 bug 면,  'bug_tracking' 리턴
    elif intent == 'bug':
        return 'bug_tracking'
    # 나머지는, 'draft_response' 리턴
    else:
        return 'draft_response'

#### 검색 및 추적 노드

In [ ]:
# RAG로 문서 찾는 노드
def search_document_node(state: EmailAgentState):
    classification = state['classification']
    # RAG에 사용할 가짜 쿼리
    query = f'{classification['intent']} - {classification['topic']}'
    print(f'----{query} 를 사용해 DB 검색중입니다...----')
    # 실제로는 vectorstore.search(query) 같은 코드를 실행하고
    search_result = [
        "Reset password via Settings > Security > Change Password",
        "Password must be at least 12 characters",
        "Include uppercase, lowercase, numbers, and symbols"
    ]
    return {'search_result': search_result}


# Github Issue 에서 해당하는 버그 찾아오는 노드
def bug_tracking_node(state: EmailAgentState):
    # github에서 issue 보고 대응하는 버그 issue id 가져오기를 가정
    issue_ids = ['BUG-123', 'BUG-456', 'BUG-789']
    return {'search_result': issue_ids}


#### 응답 생성 노드

### 5. 전체 조립

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(EmailAgentState)
builder.add_node(classify_intent_node)
builder.add_node(search_document_node)
builder.add_node(bug_tracking_node)

builder.add_edge(START, 'classify_intent_node')
builder.add_conditional_edges(
    'classify_intent_node',
    intent_router,
    {
        'human_review': ,
        'search_document': 'search_document_node',
        'bug_tracking': 'bug_tracking_node',
        'draft_response': ,
    }
)

graph = builder.compile()
graph